**PREPARACIÓN DEL DATASET DE EMOCIONES**

El modelo biométrico propio clasifica **7 expresiones**: las 6 de FER-2013  `angry`, `fear`, `happy`, `neutral`, `sad`, `surprise` más una clase nueva, **`duckface`**,

| Origen | Clases | Formato |
|---|---|---|
| FER-2013 (Kaggle) | angry, fear, happy, neutral, sad, surprise | 48x48, gris, cara ya recortada |
| Selfie-dataset | duckface | fotos de móvil en color → recortadas con el detector DNN |
| Roboflow `middle-tongue-duckface` | duckface | caras ya recortadas, en color, con aumento de Roboflow (rotaciones con esquinas negras) y anotaciones YOLO → gris 48x48 |

Decisiones sobre el dataset:
- **Formato común**: las caras nuevas se convierten al formato FER para que todas las clases lleguen igual al modelo.
- **Mismo recorte que en la webcam**: el recorte se hace con `comun.recortar_cara`, para que el encuadre de entrenamiento e inferencia sea idéntico.

In [ ]:
import os, random, shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from comun import DetectorCaras, recortar_cara, EXTENSIONES
from comun import EMOCIONES

detector = DetectorCaras(conf_min=0.5)

**1. DUCKFACE A PARTIR DEL SELFIE-DATASET**

`Selfie-dataset.tar.gz` contiene 46.836 selfies con 36 atributos binarios por imagen; la columna 23 es `duck_face`. Para cada selfie positivo:
1. Se detecta la cara con el DNN (ResNet10-SSD) y se toma la más confiada.
2. Se recorta con `recortar_cara` (5 % hacia dentro, igual que la webcam).
3. Se pasa a gris y se reduce a 48x48 con `INTER_AREA`.
4. Se reparte 80 % train / 20 % test con semilla fija.

Las imágenes en las que el detector no encuentra cara se descartan.

In [ ]:
LABELS = "dataset/Selfie-dataset/Selfie-dataset/selfie_dataset.txt"
IMGS = "dataset/Selfie-dataset/Selfie-dataset/images"
COL_DUCK = 23           # columna duck_face
TEST = 0.2
SEED = 42
REGENERAR = False       # True = borra dataset/{train,test}/duckface y la vuelve a generar


def cara_selfie(ruta):
    img = cv2.imread(ruta)
    if img is None:
        return None
    caja = detector.detectar(img)
    if not caja:
        return None
    x, y, w, h, _ = max(caja, key=lambda c: c[4])       # la detección más confiada
    cara = recortar_cara(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (x, y, w, h))
    return None if cara is None else cv2.resize(cara, (48, 48), interpolation=cv2.INTER_AREA)


if not Path(LABELS).exists():
    print("No está descomprimido el Selfie-dataset: se omite este paso.")
elif Path("dataset/train/duckface").exists() and not REGENERAR:
    print("dataset/train/duckface ya existe (REGENERAR = False): se omite este paso.")
else:
    df = pd.read_csv(LABELS, sep=r"\s+", header=None)
    nombres = [str(n) if str(n).lower().endswith((".jpg", ".jpeg", ".png")) else f"{n}.jpg"
               for n in df.loc[df[COL_DUCK] == 1, 0]]
    random.seed(SEED)
    random.shuffle(nombres)
    n_test = int(round(len(nombres) * TEST))
    for split, lista in {"test": nombres[:n_test], "train": nombres[n_test:]}.items():
        destino = Path("dataset", split, "duckface")
        if REGENERAR and destino.exists():
            shutil.rmtree(destino)
        destino.mkdir(parents=True, exist_ok=True)
        ok = descartadas = 0
        for nombre in lista:
            cara = cara_selfie(os.path.join(IMGS, nombre))
            if cara is None:
                descartadas += 1
                continue
            cv2.imwrite(str(destino / (Path(nombre).stem + ".png")), cara)
            ok += 1
        print(f"{split}/duckface: {ok} guardadas | {descartadas} sin cara o ilegibles")

**2. RECUENTO FINAL POR CLASE**

Se comprueba que las carpetas coinciden con `comun.EMOCIONES` y cuántas imágenes tiene cada clase. El desbalance se compensa después en el entrenamiento con `class_weight`.

In [ ]:
filas = {}
for split in ("train", "test"):
    carpetas = sorted(p.name for p in Path("dataset", split).iterdir() if p.is_dir() and not p.name.startswith("_"))
    if carpetas != sorted(EMOCIONES):
        print(f"[aviso] dataset/{split} tiene {carpetas}, se esperaba {sorted(EMOCIONES)}")
    filas[split] = {e: sum(1 for f in Path("dataset", split, e).iterdir() if f.suffix.lower() in EXTENSIONES)
                    for e in EMOCIONES}
tabla = pd.DataFrame(filas)
tabla["train_%"] = (100 * tabla["train"] / tabla["train"].sum()).round(1)
display(tabla)

**3. DEGRADACIÓN FOTOMÉTRICA DEL TRAIN (CONTRASTE, DESENFOQUE Y RUIDO)**

Las imágenes de `duckface` vienen del Selfie-dataset y las del resto de clases de FER-2013. Las de `duckface` son más claras y menos nítidas (varianza del laplaciano ≈ 1100 frente a ≈ 2150), así que el modelo puede aprender "imagen suave y clara ⇒ duckface" en vez de la expresión, y en directo clasifica caras neutras como `duckface`.

Para que brillo, contraste y nitidez dejen de ser una pista, se degrada una fracción aleatoria (`FRACCION`) del train de **todas las clases** con:
- **Contraste:** ±25 %.
- **Desenfoque gaussiano:** sigma entre 0,3 y 1,2.
- **Ruido:** 0,03.

Se hace una sola vez, con semilla fija, y vale para todos los modelos (01–06). **El test no se toca.**

> Los originales se guardan en `dataset/_train_original`, y la degradación siempre parte de esa copia, así que se puede repetir sin degradar dos veces. Si se regenera duckface (`REGENERAR = True`), hay que borrar `dataset/_train_original` antes de volver a degradar. Después de ejecutarlo con `DEGRADAR = True` hay que reentrenar los modelos y repetir la comparativa.

In [ ]:
DEGRADAR = True       # True = degrada dataset/train (siempre partiendo de dataset/_train_original)
FRACCION = 0.6         # fracción de imágenes de train que se degrada
SEED_DEG = 42


def degradar_imagen(img, rng):
    """Contraste, desenfoque gaussiano y ruido aleatorios sobre una imagen gris uint8."""
    f = img.astype(np.float32)
    media = f.mean()
    f = (f - media) * rng.uniform(0.75, 1.25) + media            # contraste ±25 %
    f = cv2.GaussianBlur(f, (5, 5), rng.uniform(0.3, 1.2))       # desenfoque, sigma 0,3-1,2
    f = f + rng.normal(0, 0.03 * 255, f.shape)                   # ruido, desviación 0,03
    return np.clip(f, 0, 255).astype(np.uint8)


def nitidez(carpeta, n=300):
    """Varianza media del laplaciano de las n primeras imágenes de la carpeta."""
    rutas = [r for r in sorted(Path(carpeta).iterdir()) if r.suffix.lower() in EXTENSIONES][:n]
    return float(np.mean([cv2.Laplacian(cv2.imread(str(r), 0), cv2.CV_64F).var() for r in rutas]))


def degradar_train(fraccion=FRACCION, seed=SEED_DEG,
                   train=Path("dataset/train"), orig=Path("dataset/_train_original")):
    if not orig.exists():                       # primera vez: se guardan los originales
        shutil.copytree(train, orig)
    rng = np.random.default_rng(seed)
    n_deg = n_tot = 0
    for carpeta in sorted(p for p in orig.iterdir() if p.is_dir()):
        destino = train / carpeta.name
        destino.mkdir(parents=True, exist_ok=True)
        for r in sorted(carpeta.iterdir()):
            if r.suffix.lower() not in EXTENSIONES:
                continue
            img = cv2.imread(str(r), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            if rng.random() < fraccion:
                img = degradar_imagen(img, rng)
                n_deg += 1
            cv2.imwrite(str(destino / r.name), img)
            n_tot += 1
    return n_deg, n_tot


if DEGRADAR:
    n_deg, n_tot = degradar_train()
    print(f"Degradadas {n_deg} de {n_tot} imágenes de train (originales en dataset/_train_original)")
    print("Nitidez (varianza del laplaciano), original -> degradado:")
    for e in EMOCIONES:
        print(f"  {e:9s} {nitidez(Path('dataset/_train_original', e)):7.0f} -> {nitidez(Path('dataset/train', e)):7.0f}")
else:
    print("DEGRADAR = False: no se modifica el dataset.")